In [1]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from google import genai


In [ ]:
load_dotenv("../.env", override=True)

True

In [3]:
processed_dir = Path("processed")

all_pairs = []

processed_files = list(processed_dir.glob("*.json"))

for filepath in processed_files:
    with open(filepath, encoding="utf-8") as f:
        data = json.load(f)
    
    for pair in data["qa_pairs"]:
        all_pairs.append({
            "symbol": data["symbol"],
            "quarter": data["quarter"],
            "year": data["year"],
            "analyst": pair["analyst"],
            "question": pair["question"],
            "answer": pair["answer"]
        })

print("Total Q&A pairs loaded:", len(all_pairs))
print()
print("First pair:")
print(all_pairs[0])

Total Q&A pairs loaded: 2855

First pair:
{'symbol': 'AAPL', 'quarter': 1, 'year': 2021, 'analyst': 'Samik Chatterjee', 'question': "Congrats on the record quarter from my side as well. I guess I wanted to start off with iPhone sales. I think in -- general impression we have is China and North America have more robust 5G infrastructure. I just wanted to see kind of what are you seeing in terms of customer engagement or velocity of sales for iPhone in Europe, where I think the general impression is that service providers haven't rolled out robust 5G services. Is that something that's impacting customer interest in the latest lineup in the region? And I have a follow-up.", 'answer': "If you look at the 5G rollout in Europe, it's true that Europe is not in the place of -- certainly nowhere close to where China is and nowhere close to the U.S. either. But there are other regions that 5G is -- that has very good coverage, like Korea is an example. And so the the world, I would describe it r

In [48]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [5]:
def label_batch(pairs_batch):
    numbered_pairs = ""
    for i, pair in enumerate(pairs_batch):
        question_short = pair["question"][:400]
        answer_short = pair["answer"][:600]
        numbered_pairs = numbered_pairs + f"\n\nPAIR {i+1}:\nQUESTION: {question_short}\nANSWER: {answer_short}"
    
    prompt = f"""You are a financial analyst evaluating earnings call answers.

For each pair below, decide if the executive directly answered the question (DIRECT) or avoided it (EVASIVE).

DIRECT means: specific, on-topic answer addressing what was asked. Search for following sign in answer for DIRECT
- Gives numbers, guidance, timeline, reason, explanation
- Directly addresses the question

EVASIVE: vague commitments, redirecting to unrelated topics, refusing specifics, corporate boilerplate without substance. Search for following sign in answer for EVASIVE
- Avoids requested specifics
- Gives generic corporate language
- Says "we remain confident"
- Says "too early to comment"
- Changes topic

When uncertain and in doubt, choose EVASIVE.
IMPORTANT NOTE: Evaluate EACH pair independently. Do NOT compare pairs with one another. For every pair, ignore all other pairs.

{numbered_pairs}

Respond with ONLY a JSON array, one entry per pair, in the same order, like this:
[{{"pair": 1, "label": "DIRECT", "confidence": 0.9}}, {{"pair": 2, "label": "EVASIVE", "confidence": 0.8}}]

No other text, just the JSON array."""

    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a strict financial analyst. Follow the labeling rules exactly."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)
    
    text = response.choices[0].message.content.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    
    try:
        results = json.loads(text)
        return results
    except Exception as e:
        print("JSON Error:", e)
        print(text)
        return []

In [6]:
def make_batches(items, batch_size):
    batches = []
    for i in range(0, len(items), batch_size):
        batch = items[i:i + batch_size]
        batches.append(batch)
    return batches


In [8]:
BATCH_SIZE = 8

all_batches = make_batches(all_pairs, BATCH_SIZE)

print("Total pairs:", len(all_pairs))
print("Batch size:", BATCH_SIZE)
print("Total batches:", len(all_batches))

Total pairs: 2855
Batch size: 8
Total batches: 357


In [9]:
import time

In [11]:
all_labeled_pairs = []
for batch_num, batch in enumerate(all_batches):
    try:
        results = label_batch(batch)
        
        for i, result in enumerate(results):
            original_pair = batch[i]
            all_labeled_pairs.append({
                "symbol": original_pair["symbol"],
                "quarter": original_pair["quarter"],
                "year": original_pair["year"],
                "analyst": original_pair["analyst"],
                "question": original_pair["question"],
                "answer": original_pair["answer"],
                "label": result["label"],
                "confidence": result["confidence"]
            })
        
        with open("evasion_labels_partial.json", "w", encoding="utf-8") as f:
            json.dump(all_labeled_pairs, f, indent=2, ensure_ascii=False)
        
        print("Batch", batch_num + 1, "of", len(all_batches), "- OK -", len(results), "pairs labeled - saved")
        
    except Exception as e:
        error_text = str(e)
        if "rate_limit_exceeded" in error_text or "429" in error_text:
            print()
            print("Daily limit reached. Stopping here for today.")
            break
        else:
            print("Batch", batch_num + 1, "- FAILED (non rate-limit error) -", error_text)
    
    time.sleep(2.5)

print()
print("Total labeled:", len(all_labeled_pairs))
print("Saved to evasion_labels_partial.json")

Batch 1 of 136 - OK - 8 pairs labeled - saved
Batch 2 of 136 - OK - 8 pairs labeled - saved
Batch 3 of 136 - OK - 8 pairs labeled - saved
Batch 4 of 136 - OK - 8 pairs labeled - saved
Batch 5 of 136 - OK - 8 pairs labeled - saved
Batch 6 of 136 - OK - 8 pairs labeled - saved
Batch 7 of 136 - OK - 8 pairs labeled - saved
Batch 8 of 136 - OK - 8 pairs labeled - saved
Batch 9 of 136 - OK - 8 pairs labeled - saved
Batch 10 of 136 - OK - 8 pairs labeled - saved
Batch 11 of 136 - OK - 8 pairs labeled - saved
Batch 12 of 136 - OK - 8 pairs labeled - saved
Batch 13 of 136 - OK - 8 pairs labeled - saved
Batch 14 of 136 - OK - 8 pairs labeled - saved
Batch 15 of 136 - OK - 8 pairs labeled - saved
Batch 16 of 136 - OK - 8 pairs labeled - saved
Batch 17 of 136 - OK - 8 pairs labeled - saved
Batch 18 of 136 - OK - 8 pairs labeled - saved
Batch 19 of 136 - OK - 8 pairs labeled - saved
Batch 20 of 136 - OK - 8 pairs labeled - saved
Batch 21 of 136 - OK - 8 pairs labeled - saved
Batch 22 of 136 - OK -

In [12]:
already_labeled_keys = set()
for pair in all_labeled_pairs:
    key = pair["symbol"] + "_" + str(pair["quarter"]) + "_" + str(pair["year"]) + "_" + pair["question"][:50]
    already_labeled_keys.add(key)

remaining_pairs = []
for pair in all_pairs:
    key = pair["symbol"] + "_" + str(pair["quarter"]) + "_" + str(pair["year"]) + "_" + pair["question"][:50]
    if key not in already_labeled_keys:
        remaining_pairs.append(pair)

print("Already labeled:", len(all_labeled_pairs))
print("Remaining to label:", len(remaining_pairs))

Already labeled: 1082
Remaining to label: 1773


In [13]:
remaining_batches = make_batches(remaining_pairs, BATCH_SIZE)

print("Remaining batches:", len(remaining_batches))

Remaining batches: 222


In [30]:
for batch_num, batch in enumerate(remaining_batches):
    try:
        results = label_batch(batch)
        
        for i, result in enumerate(results):
            original_pair = batch[i]
            all_labeled_pairs.append({
                "symbol": original_pair["symbol"],
                "quarter": original_pair["quarter"],
                "year": original_pair["year"],
                "analyst": original_pair["analyst"],
                "question": original_pair["question"],
                "answer": original_pair["answer"],
                "label": result["label"],
                "confidence": result["confidence"]
            })
        
        with open("evasion_labels_partial.json", "w", encoding="utf-8") as f:
            json.dump(all_labeled_pairs, f, indent=2, ensure_ascii=False)
        
        print("Batch", batch_num + 1, "of", len(remaining_batches), "- OK -", len(results), "pairs labeled - saved")
        
    except Exception as e:
        error_text = str(e)
        if "rate_limit_exceeded" in error_text or "429" in error_text:
            print()
            print("Daily limit reached. Stopping here for today.")
            break
        else:
            print("Batch", batch_num + 1, "- FAILED (non rate-limit error) -", error_text)
    
    time.sleep(2.5)

print()
print("Total labeled so far:", len(all_labeled_pairs))
print("Saved to evasion_labels_partial.json")

Batch 1 of 27 - OK - 8 pairs labeled - saved
Batch 2 of 27 - OK - 8 pairs labeled - saved
Batch 3 of 27 - OK - 8 pairs labeled - saved
Batch 4 of 27 - OK - 8 pairs labeled - saved
Batch 5 of 27 - OK - 8 pairs labeled - saved
Batch 6 of 27 - OK - 8 pairs labeled - saved
Batch 7 of 27 - OK - 8 pairs labeled - saved
Batch 8 of 27 - OK - 8 pairs labeled - saved
Batch 9 of 27 - OK - 8 pairs labeled - saved
Batch 10 of 27 - OK - 8 pairs labeled - saved
Batch 11 of 27 - OK - 8 pairs labeled - saved
Batch 12 of 27 - OK - 8 pairs labeled - saved
Batch 13 of 27 - OK - 8 pairs labeled - saved
Batch 14 of 27 - OK - 8 pairs labeled - saved
Batch 15 of 27 - OK - 8 pairs labeled - saved
Batch 16 of 27 - OK - 8 pairs labeled - saved
Batch 17 of 27 - OK - 8 pairs labeled - saved
Batch 18 of 27 - OK - 8 pairs labeled - saved
Batch 19 of 27 - OK - 8 pairs labeled - saved
Batch 20 of 27 - OK - 8 pairs labeled - saved
Batch 21 of 27 - OK - 8 pairs labeled - saved
Batch 22 of 27 - OK - 8 pairs labeled - sav

In [11]:
if os.path.exists("evasion_labels_partial.json"):
    with open("evasion_labels_partial.json", encoding="utf-8") as f:
        all_labeled_pairs = json.load(f)
    print("Loaded", len(all_labeled_pairs), "previously labeled pairs from disk")
else:
    all_labeled_pairs = []
    print("No previous file found, starting fresh")

Loaded 1082 previously labeled pairs from disk


In [34]:
with open("evasion_labels_partial.json", encoding="utf-8") as f:
    labeled_data = json.load(f)

greeting_phrases = [
    "thanks",
    "thank you",
    "good morning",
    "good afternoon",
    "good evening",
    "congratulations",
    "that's all from me",
    "appreciate it",
    "welcome back",
]

question_words = [
    "what",
    "why",
    "how",
    "when",
    "where",
    "which",
    "who",
    "can",
    "could",
    "would",
    "should",
    "do you",
    "did you",
    "are you",
    "is there",
    "will you",
]

not_usefull = []
usefull = []

for pair in labeled_data:
    q = pair["question"].lower().strip()

    is_short = len(q.split()) < 8

    has_greeting = any(
        phrase in q
        for phrase in greeting_phrases
    )

    has_question_word = any(
        word in q
        for word in question_words
    )

    if (is_short and has_greeting) or not has_question_word:
        not_usefull.append(pair)
    else: 
        usefull.append(pair)



with open("evasion_labels_usefullPair.json", "w", encoding="utf-8") as f:
    json.dump(usefull, f, indent=2, ensure_ascii=False)
print("Saved", len(usefull), "genuine pairs to evasion_labels_usefUllPair.json")

with open("evasion_removed_pleasantries.json", "w", encoding="utf-8") as f:
    json.dump(not_usefull, f, indent=2, ensure_ascii=False)
print("Saved", len(not_usefull), "removed pairs to evasion_removed_pleasantries.json")

print(f"Total pairs: {len(labeled_data)}")
print(f"Not usefull pairs: {len(not_usefull)}")
print(f"Usefull pairs: {len(usefull)}")
print()

for i, p in enumerate(not_usefull[:50], 1):
    print("=" * 80)
    print(f"SUSPECT #{i}")
    print(f"Label: {p['label']}")
    print()
    print("QUESTION:")
    print(p["question"])
    print()
    print("ANSWER:")
    print(p["answer"][:300])
    print()

Saved 871 genuine pairs to evasion_labels_usefUllPair.json
Saved 211 removed pairs to evasion_removed_pleasantries.json
Total pairs: 1082
Not usefull pairs: 211
Usefull pairs: 871

SUSPECT #1
Label: EVASIVE

QUESTION:
Wonderful. Thank you, Tim.

ANSWER:
Yes. Thank you, Mike. Operator, can we have the next question please?

SUSPECT #2
Label: EVASIVE

QUESTION:
Thank you so much, Tim.

ANSWER:
Thanks, Wamsi. Operator, we'll take the next question please.

SUSPECT #3
Label: EVASIVE

QUESTION:
Perfect. That's a really good perspective to have. Thank you very much.

ANSWER:
Thank you, Amit. Operator, can we have the next question please?

SUSPECT #4
Label: EVASIVE

QUESTION:
Thank you.

ANSWER:
Thanks, Aaron. Operator, can we have the next question please?

SUSPECT #5
Label: EVASIVE

QUESTION:
Great. Thanks a lot Tim. Very interesting to hear. Thank you.

ANSWER:
Thank you, Krish. Operator, can we have the next question, please?

SUSPECT #6
Label: EVASIVE

QUESTION:
Got it. Thanks, guys.

A

In [6]:
with open("../data/evasion_labels_partial.json", encoding="utf-8") as f:
    all_labeled_pairs = json.load(f)

print("Already labeled:", len(all_labeled_pairs))

Already labeled: 1082


In [8]:
already_labeled_keys = set()
for pair in all_labeled_pairs:
    key = pair["symbol"] + "_" + str(pair["quarter"]) + "_" + str(pair["year"]) + "_" + pair["question"][:50]
    already_labeled_keys.add(key)

remaining_pairs = []
for pair in all_pairs:
    key = pair["symbol"] + "_" + str(pair["quarter"]) + "_" + str(pair["year"]) + "_" + pair["question"][:50]
    if key not in already_labeled_keys:
        remaining_pairs.append(pair)

print("Already labeled:", len(all_labeled_pairs))
print("Remaining to label:", len(remaining_pairs))

Already labeled: 1082
Remaining to label: 1773


In [14]:
greeting_phrases = [
    "thanks",
    "thank you",
    "good morning",
    "good afternoon",
    "good evening",
    "congratulations",
    "that's all from me",
    "appreciate it",
    "welcome back",
]
question_words = [
    "what",
    "why",
    "how",
    "when",
    "where",
    "which",
    "who",
    "can",
    "could",
    "would",
    "should",
    "do you",
    "did you",
    "are you",
    "is there",
    "will you",
]

def is_genuine_pair(pair):
    q = pair["question"].lower().strip()
    is_short = len(q.split()) < 8
    has_greeting = any(phrase in q for phrase in greeting_phrases)
    has_question_word = any(word in q for word in question_words)
    
    if (is_short and has_greeting) or not has_question_word:
        return False
    return True


genuine_remaining_pairs = [p for p in remaining_pairs if is_genuine_pair(p)]

print("Remaining pairs before filtering:", len(remaining_pairs))
print("Genuine remaining pairs (to be labeled):", len(genuine_remaining_pairs))
print("Filtered out as pleasantries:", len(remaining_pairs) - len(genuine_remaining_pairs))

Remaining pairs before filtering: 1773
Genuine remaining pairs (to be labeled): 1411
Filtered out as pleasantries: 362


In [50]:
with open("../data/evasion_labels_usefullPair.json", encoding="utf-8") as f:
    all_labeled_pairs = json.load(f)

print("Already labeled:", len(all_labeled_pairs))

Already labeled: 2282


In [51]:
already_processed_keys = set()

for pair in all_labeled_pairs:
    key = pair["symbol"] + "_" + str(pair["quarter"]) + "_" + str(pair["year"]) + "_" + pair["question"][:50]
    already_processed_keys.add(key)

print("Total already-processed keys :", len(already_processed_keys))

remaining_pairs = []
for pair in genuine_remaining_pairs:
    key = pair["symbol"] + "_" + str(pair["quarter"]) + "_" + str(pair["year"]) + "_" + pair["question"][:50]
    if key not in already_processed_keys:
        remaining_pairs.append(pair)

print("Remaining to process (truly new):", len(remaining_pairs))

Total already-processed keys : 2282
Remaining to process (truly new): 0


In [52]:
remaining_batches = make_batches(remaining_pairs, BATCH_SIZE)

print("Remaining batches:", len(remaining_batches))

Remaining batches: 0


In [49]:
for batch_num, batch in enumerate(remaining_batches):
    try:
        results = label_batch(batch)
        
        for i, result in enumerate(results):
            original_pair = batch[i]
            all_labeled_pairs.append({
                "symbol": original_pair["symbol"],
                "quarter": original_pair["quarter"],
                "year": original_pair["year"],
                "analyst": original_pair["analyst"],
                "question": original_pair["question"],
                "answer": original_pair["answer"],
                "label": result["label"],
                "confidence": result["confidence"]
            })
        
        with open("../data/evasion_labels_usefullPair.json", "w", encoding="utf-8") as f:
            json.dump(all_labeled_pairs, f, indent=2, ensure_ascii=False)
        
        print("Batch", batch_num + 1, "of", len(remaining_batches), "- OK -", len(results), "pairs labeled - saved")
        
    except Exception as e:
        error_text = str(e)
        if "rate_limit_exceeded" in error_text or "429" in error_text:
            print()
            print("Daily limit reached. Stopping here for today.")
            break
        else:
            print("Batch", batch_num + 1, "- FAILED (non rate-limit error) -", error_text)
    
    time.sleep(2.5)

print()
print("Total labeled so far:", len(all_labeled_pairs))
print("Saved to evasion_labels_usefullPair.json")

Batch 1 of 36 - OK - 8 pairs labeled - saved
Batch 2 of 36 - OK - 8 pairs labeled - saved
Batch 3 of 36 - OK - 8 pairs labeled - saved
Batch 4 of 36 - OK - 8 pairs labeled - saved
Batch 5 of 36 - OK - 8 pairs labeled - saved
Batch 6 of 36 - OK - 8 pairs labeled - saved
Batch 7 of 36 - OK - 8 pairs labeled - saved
Batch 8 of 36 - OK - 8 pairs labeled - saved
Batch 9 of 36 - OK - 8 pairs labeled - saved
Batch 10 of 36 - OK - 8 pairs labeled - saved
Batch 11 of 36 - OK - 8 pairs labeled - saved
Batch 12 of 36 - OK - 8 pairs labeled - saved
Batch 13 of 36 - OK - 8 pairs labeled - saved
Batch 14 of 36 - OK - 8 pairs labeled - saved
Batch 15 of 36 - OK - 8 pairs labeled - saved
Batch 16 of 36 - OK - 8 pairs labeled - saved
Batch 17 of 36 - OK - 8 pairs labeled - saved
Batch 18 of 36 - OK - 8 pairs labeled - saved
Batch 19 of 36 - OK - 8 pairs labeled - saved
Batch 20 of 36 - OK - 8 pairs labeled - saved
Batch 21 of 36 - OK - 8 pairs labeled - saved
Batch 22 of 36 - OK - 8 pairs labeled - sav

In [54]:
with open("../data/evasion_labels_usefullPair.json", encoding="utf-8") as f:
    final_labeled = json.load(f)

print("Total pairs in file:", len(final_labeled))
print()

for pair in final_labeled[-3:]:
    print("Symbol:", pair["symbol"], "| Quarter:", pair["quarter"], "| Year:", pair["year"])
    print("Question:", pair["question"][:500])
    print()
    print("Answer:", pair["answer"][:500])
    print()
    print("Label:", pair["label"], "| Confidence:", pair["confidence"])
    print("=" * 80)

Total pairs in file: 2282

Symbol: WTW | Quarter: 4 | Year: 2022
Question: Got it. That’s helpful. And then my second question is more on a high level. At your last Investor Day, you talked about kind of moving upstream to larger accounts. Can you just give us an update on how that’s progressing? And how that’s contemplated in your mid-single digit organic growth guidance for this year?

Answer: We’re happy with how we continue to be a large account reference there was principally across the R&D portfolio, right? Our existing HWC business skews large accounts to begin with. With respect to how we’re progressing in large market, I think I’d just point to the fact that we’re growing and large market is very much a part of that. Some of the talent we brought on focus is there. But we think we play well across all market segments, and that’s a strength we have for us.

Label: EVASIVE | Confidence: 0.8
Symbol: WTW | Quarter: 4 | Year: 2022
Question: Good morning, and thanks for taking my qu